# Nghe — build the audio

Run the cells top to bottom. Everything happens on Google's machines, not yours.

**Before you start:** click **Runtime -> Change runtime type -> T4 GPU -> Save.**
Free tier is fine. On CPU this still works but takes several times longer.

## 1. Connect your Google Drive

Colab disconnects after a while and forgets everything. Working inside Drive means
a disconnect costs you nothing — you just re-run from here and it picks up where it stopped.

A permission popup will appear. Accept it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/nghe'
os.makedirs(WORK, exist_ok=True)
print('working in', WORK)

## 2. Pull in your repo

Set `REPO` to your own repository below.

For the token: on GitHub go to **Settings -> Developer settings -> Personal access tokens ->
Fine-grained tokens -> Generate new token**. Give it access to this one repository, and under
**Repository permissions** set **Contents** to **Read and write**. Copy the token.

Then in Colab click the **key icon** in the left sidebar, add a secret named `GITHUB_TOKEN`,
paste the token in, and turn on notebook access. It stays private to your account.

In [ ]:
REPO  = 'your-username/nghe'        # <-- change this
EMAIL = 'you@example.com'           # <-- and this
NAME  = 'Your Name'                 # <-- and this

from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')

import os, subprocess
REPO_DIR = os.path.join(WORK, REPO.split('/')[-1])
url = f'https://{TOKEN}@github.com/{REPO}.git'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git','-C',REPO_DIR,'pull','--rebase'], check=False)
else:
    subprocess.run(['git','clone',url,REPO_DIR], check=True)

subprocess.run(['git','-C',REPO_DIR,'config','user.email',EMAIL], check=True)
subprocess.run(['git','-C',REPO_DIR,'config','user.name',NAME], check=True)
subprocess.run(['git','-C',REPO_DIR,'remote','set-url','origin',url], check=True)
os.chdir(REPO_DIR)
print('repo ready at', REPO_DIR)
print(os.listdir('.'))

## 3. Install VieNeu

VieNeu is the Vietnamese text-to-speech model. This takes a couple of minutes the first time.
You will not need to run it again unless the Colab session is brand new.

In [ ]:
!pip install -q vieneu
!ffmpeg -version | head -1

## 4. Choose a voice

This lists the built-in voices. Pick a **Southern** one — the description tells you which.
Copy its ID into the next cell.

In [ ]:
from vieneu import Vieneu
tts = Vieneu()
for desc, name in tts.list_preset_voices():
    print(f'{name:24s} {desc}')

## 5. Listen to 40 clips before committing

Do not skip this. If the voice mangles isolated syllables you want to find out now,
not after a six-hour run.

Listen to a handful. Check especially that the tone you hear matches the tone mark written.
If they sound clipped or trail off oddly, set `CARRIER` to `'Từ {} .'` and re-run —
that puts the syllable in a short frame and cuts the frame back off.

In [ ]:
VOICE   = 'PUT-THE-VOICE-ID-HERE'   # <-- from the list above
CARRIER = None                      # or 'Từ {} .' if bare syllables sound wrong

!python generate_clips.py \
    --manifest vietnamese_clip_manifest.csv \
    --outdir audio --voice "$VOICE" --limit 40

import glob, IPython.display as ipd, csv
rows = {r['clip_id']: r['syllable'] for r in
        csv.DictReader(open('vietnamese_clip_manifest.csv', encoding='utf-8-sig'))}
for p in sorted(glob.glob('audio/*.mp3'))[:12]:
    cid = p.split('/')[-1].split('.')[0]
    print(cid, rows.get(cid))
    ipd.display(ipd.Audio(p))

## 6. The full run

This is the long one. It is **resumable** — if Colab disconnects, just re-run this cell
and it skips everything already done. Keep the tab open; Colab stops free sessions that
look idle.

In [ ]:
!python generate_clips.py \
    --manifest vietnamese_clip_manifest.csv \
    --outdir audio --voice "$VOICE"

import glob
print(len(glob.glob('audio/*.mp3')), 'clips')
!du -sh audio

## 7. Check the clips are actually right

PhoWhisper is a Vietnamese speech recogniser. This plays every clip back into it and flags
any that do not come back as the syllable they were meant to be — which is how you catch a
clip with the wrong tone before it teaches you the wrong tone.

Expect some false alarms on rare syllables: the recogniser is not perfect either.
`tone_mismatch` rows are the ones worth listening to.

In [ ]:
!pip install -q librosa transformers
!python qc_clips.py --manifest vietnamese_clip_manifest.csv --outdir audio

import pandas as pd
qc = pd.read_csv('qc_report.csv')
display(qc['status'].value_counts())
display(qc[qc.status == 'tone_mismatch'].head(20))

### Listen to the flagged ones

Run this to hear what the recogniser objected to. If a clip really is wrong, delete it —
the app falls back to the browser voice for anything missing, and you can regenerate later.

In [ ]:
import IPython.display as ipd
bad = qc[qc.status != 'ok'].head(15)
for _, r in bad.iterrows():
    print(f"{r.syllable}  ->  heard '{r.heard}'  ({r.status})")
    ipd.display(ipd.Audio(f"audio/{r.clip_id}.mp3"))

## 8. Publish

Rebuilds the app's data file, then pushes everything to GitHub. A minute or two later
your site is live.

In [ ]:
!python build_data.py \
    --items vietnamese_drill_items.csv \
    --clips vietnamese_clip_manifest.csv \
    --out data.json

!git add -A && git -c commit.gpgsign=false commit -m 'Add generated audio' || echo 'nothing new to commit'
!git push origin HEAD

---

**Re-running later.** To add more clips (say you raised the vocabulary range), re-run
cells 2, 4, 6 and 8. Step 6 only makes what is missing.